<img src="https://raw.githubusercontent.com/autonomousvision/navsim/main/assets/navsim_transparent.png" alt="drawing" width="800"/>

# NAVSIM Visualization Tutorial

This notebook introduces the core plots used to visualize driving scenes in NAVSIM. All plots are created with `matplotlib` and can be reused in custom scripts.

## Table of Contents
1. [Visualization code structure](#structure)
2. [Config](#config)
3. [Birds-Eye-View](#bev)
4. [Cameras](#camera)
5. [Creating custom plots](#custom)
6. [Creating GIFs](#gifs)


## Visualization code structure <a name="structure"></a>

Visualization code is split by responsibility:

- `navsim/visualization/config.py`: shared colors, figure sizes, BEV layers, and agent styles.
- `navsim/visualization/bev.py`: low-level BEV primitives for maps, annotations, ego boxes, trajectories, and LiDAR.
- `navsim/visualization/camera.py`: camera image rendering, annotation projection, LiDAR projection, and trajectory projection.
- `navsim/visualization/plots.py`: figure-level helpers such as BEV/camera grids and `frame_plot_to_gif`.
- `navsim/evaluate/eval_visualization.py`: evaluation-time static BEV/camera comparison helpers.
- `navsim/evaluate/render_top1_from_detail.py`: CLI renderer for `detail.json` outputs, including static BEV/front-camera figures and model-trajectory GIFs with future GT agents.
- `navsim/evaluate/render_gt_gif_from_tokens.py`: CLI/API renderer for GT-only BEV GIFs from selected NAVSIM token/log pairs.
- `navsim/evaluate/chainflow_vis.sh`: batch wrapper around `render_top1_from_detail.py` for the ChainFlow-VLA qualitative token set.
- `scripts/viz/render_gt_gif_pairs.py`: project-specific wrapper that renders the qualitative token set with the shared GT GIF renderer.


## Config <a name="config"></a>

NAVSIM offers two types of plots: 
- Birds-Eye-View (BEV) plots or 
- Camera plots. 

The LiDAR sensor can be visualized either in BEV or in camera images. All plots have a global configuration in [`navsim/visualization/config.py`](https://github.com/autonomousvision/navsim/blob/main/navsim/navsim/visualization/config.py). In this Python file, you can configure all colors or dimensions. The LiDAR point cloud can be colored in any colormap, showing the distance to the ego vehicle or the height of each point. In this tutorial, we first instantiate a `SceneFilter` and `SceneLoader` from the mini split.

In [ ]:
import os
from pathlib import Path

import hydra
from hydra.utils import instantiate
import numpy as np
import matplotlib.pyplot as plt

from navsim.common.dataloader import SceneLoader
from navsim.common.dataclasses import SceneFilter, SensorConfig

SPLIT = "mini"  # ["mini", "test", "trainval"]
FILTER = "all_scenes"

hydra.initialize(config_path="../navsim/planning/script/config/common/scene_filter")
cfg = hydra.compose(config_name=FILTER)
scene_filter: SceneFilter = instantiate(cfg)
openscene_data_root = Path(os.getenv("OPENSCENE_DATA_ROOT"))

scene_loader = SceneLoader(
    openscene_data_root / f"navsim_logs/{SPLIT}",
    openscene_data_root / f"sensor_blobs/{SPLIT}",
    scene_filter,
    sensor_config=SensorConfig.build_all_sensors(),
)

## Birds-Eye-View <a name="bev"></a>

The Birds-Eye-View (BEV) visualization in NAVSIM is useful for overviewing the map, bounding-box annotations, or the LiDAR point cloud. In standard setting, the BEV plot includes a 64m $\times$ 64m frame centered at the rear axle of the ego vehicle (excluding LiDAR for simplicity). First, we take a random token and load a scene to visualize.

In [ ]:
token = np.random.choice(scene_loader.tokens)
scene = scene_loader.get_scene_from_token(token)

The function `plot_bev_frame` takes a `Scene` and index of the step to visualize (history or future). 

In [ ]:
from navsim.visualization.plots import plot_bev_frame

frame_idx = scene.scene_metadata.num_history_frames - 1 # current frame
fig, ax = plot_bev_frame(scene, frame_idx)
plt.show()

The function `plot_bev_with_agent` visualizes the trajectory of an <span style="color:#DE7061">agent</span> in comparison to the <span style="color:#59a14f">human vehicle operator</span> at the current frame. This notebook shows an example of the naive `ConstantVelocityAgent`:

In [ ]:
from navsim.visualization.plots import plot_bev_with_agent
from navsim.agents.constant_velocity_agent import ConstantVelocityAgent

agent = ConstantVelocityAgent()
fig, ax = plot_bev_with_agent(scene, agent)
plt.show()

## Cameras <a name="camera"></a>

The agents in NAVSIM have access to eight cameras surrounding the vehicle. The function `plot_cameras_frame` shows the cameras in a 3 $\times$ 3 grid with cameras in each direction of the ego-vehicle and the BEV plot in the center. 

In [ ]:
from navsim.visualization.plots import plot_cameras_frame

fig, ax = plot_cameras_frame(scene, frame_idx)
plt.show()

With `plot_cameras_frame_with_annotations`, you can visualize the bounding-box annotations in the camera images.

In [ ]:
from navsim.visualization.plots import plot_cameras_frame_with_annotations

fig, ax = plot_cameras_frame_with_annotations(scene, frame_idx)
plt.show()

With `plot_cameras_frame_with_lidar`, you can visualize the LiDAR point cloud in the camera images.

In [ ]:
from navsim.visualization.plots import plot_cameras_frame_with_lidar

fig, ax = plot_cameras_frame_with_lidar(scene, frame_idx)
plt.show()

## Creating custom plots <a name="custom"></a>

The plots in NAVSIM use `matplotlib` and either add elements to a `plt.Axes` object or return the full `plt.Figure`. Functions in [`navsim/visualization/`](https://github.com/autonomousvision/navsim/blob/main/navsim/navsim/visualization) can be re-used to create custom plots. In this example, we create a plot for the bounding-box annotations and the LiDAR point cloud.

In [ ]:
from navsim.visualization.plots import configure_bev_ax
from navsim.visualization.bev import add_annotations_to_bev_ax, add_lidar_to_bev_ax


fig, ax = plt.subplots(1, 1, figsize=(6, 6))

ax.set_title("Custom plot")

add_annotations_to_bev_ax(ax, scene.frames[frame_idx].annotations)
add_lidar_to_bev_ax(ax, scene.frames[frame_idx].lidar)

# configures frame to BEV view
configure_bev_ax(ax)

plt.show()

## Creating GIFs <a name="gifs"></a>

There are three common GIF workflows:

1. Use `frame_plot_to_gif` for generic frame-wise camera or BEV plots.
2. Use `render_gt_bev_gif` for GT-only BEV animations from a loaded `Scene`.
3. Use `render_top1_from_detail.py` when visualizing model outputs stored in a `proposal_top1/detail.json` file.


In [ ]:
from navsim.visualization.plots import frame_plot_to_gif

frame_indices = [idx for idx in range(len(scene.frames))]  # all loaded history + future frames
file_name = f"./{token}_camera_annotations.gif"
frame_plot_to_gif(file_name, plot_cameras_frame_with_annotations, scene, frame_indices)


### GT-only BEV GIFs

`render_gt_bev_gif` follows the same BEV animation convention as the evaluation renderer: the map is fixed in the current ego frame, the planning-time ego box is transparent, the GT ego trajectory is green, and future GT agent boxes are warped into the current ego frame.


In [ ]:
from pathlib import Path

from navsim.evaluate.render_gt_gif_from_tokens import render_gt_bev_gif

gt_gif_path = Path(f"./{token}_BEV_gt.gif")
render_gt_bev_gif(scene, gt_gif_path, gif_steps=8, gif_fps=4.0)


### Command-line rendering

For one token/log pair, call the reusable GT renderer directly. The token is the selected NAVSIM sample and `log_name` comes from `scene.scene_metadata.log_name`.


In [ ]:
# Example shell command:
# python navsim/evaluate/render_gt_gif_from_tokens.py \
#   --pair ${TOKEN},${LOG_NAME} \
#   --maps-root /data/download/maps \
#   --data-path /path/to/navsim_logs/test \
#   --sensor-blobs-path /path/to/sensor_blobs/test \
#   --output-dir /tmp/navsim_gt_gifs \
#   --gif-steps 8 \
#   --gif-fps 4


For the qualitative token set used in the ChainFlow figures, use the project-specific wrapper:


In [ ]:
# Example shell command:
# python scripts/viz/render_gt_gif_pairs.py \
#   --maps-root /data/download/maps \
#   --output-dir /data/chainflow-e2e/visualizations/gt_gifs \
#   --gif-steps 8 \
#   --gif-fps 4


### Model-output GIFs

When you have a `proposal_top1/detail.json`, use `render_top1_from_detail.py`. This overlays the model trajectory and can also render a BEV GIF with future GT agents.


In [ ]:
# Example shell command:
# python navsim/evaluate/render_top1_from_detail.py \
#   --detail-json /path/to/<log_name>/<token>/proposal_top1/detail.json \
#   --traj-field pred_traj_local \
#   --data-path /path/to/navsim_logs/test \
#   --sensor-blobs-path /path/to/sensor_blobs/test \
#   --output-dir /tmp/navsim_model_vis \
#   --image-prefix ChainFlow_${TOKEN} \
#   --render-bev-gif \
#   --gif-steps 8 \
#   --gif-fps 4 \
#   --format none
